# 1. Creating the folders- data and output


In [ ]:

# ==============================================================
# STEP 0 — Project setup: folders + environment check
# ==============================================================
from pathlib import Path
import sys

# The project will live wherever this notebook is. Confirm that's what you want:
PROJECT_ROOT = Path.cwd()
print("Project root (where the notebook is running):")
print("   ", PROJECT_ROOT)
print()

# Create the folder structure we'll use throughout:
folders = [
    "data/raw",              # downloaded 10-K HTML files
    "data/processed",        # extracted + cleaned text, feature CSVs
    "outputs",               # results, tables, figures
]
for f in folders:
    (PROJECT_ROOT / f).mkdir(parents=True, exist_ok=True)
    print("  created/verified:", f)

print("\nPython version:", sys.version.split()[0])
print("Setup done. Next: we install the libraries.")

# 2. Library Set Up

In [3]:
# ==============================================================
# STEP 1 — Install libraries
# ==============================================================
# Run once. If a line errors, paste it to me before continuing.
!pip install requests beautifulsoup4 lxml pandas numpy
!pip install nltk scikit-learn
!pip install sentence-transformers
!pip install pyphen       
# for Fog readability syllable counting

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ------------------------------ --------- 1.6/2.1 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 5.7 MB/s  0:00:00


In [4]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aradh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aradh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aradh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Step 2 — Fetch 10-K filings from SEC EDGAR

In [5]:
# ==============================================================
# CELL 2a — Panel definition & EDGAR settings
# ==============================================================
import time, json, requests
from pathlib import Path

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SEC_USER_AGENT = "Aleena aleenanotes21@gmail.com"

# The 8-bank panel. CIKs are the EDGAR company IDs.
# SVB is our subject; the rest are regional-bank peers.
# SBNY & FRC intentionally excluded (filed with FDIC, not EDGAR).
BANKS = {
    "SIVB": 719739,    # SVB Financial Group (subject; delisted Mar 2023)
    "PACW": 1102112,   # PacWest Bancorp
    "WAL":  1212545,   # Western Alliance
    "ZION": 109380,    # Zions Bancorporation
    "CMA":  28412,     # Comerica
    "KEY":  91576,     # KeyCorp
    "RF":   1281761,   # Regions Financial
    "FITB": 35527,     # Fifth Third Bancorp
}
# HBAN deliberately dropped: its Item 1A extraction fails silently (audited).

FISCAL_YEARS = list(range(2016, 2023))  # 2016–2022 (SVB's last filing is FY2022)

print(f"Panel: {len(BANKS)} banks")
print(f"Years: {FISCAL_YEARS[0]}–{FISCAL_YEARS[-1]}")
print(f"Max filings to fetch: {len(BANKS) * len(FISCAL_YEARS)}")

Panel: 8 banks
Years: 2016–2022
Max filings to fetch: 56

Before running 2b: edit SEC_USER_AGENT with your real email above.


# Data Fetching

In [6]:
# ==============================================================
# CELL 2b — Download 10-K filings from EDGAR
# ==============================================================
HEADERS = {"User-Agent": SEC_USER_AGENT}

def get_submissions(cik):
    """Fetch a company's full filing history from EDGAR."""
    url = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def find_10k_filings(submissions, target_years):
    """From the submissions JSON, pick one 10-K per target fiscal year."""
    recent = submissions["filings"]["recent"]
    forms        = recent["form"]
    report_dates = recent["reportDate"]
    accessions   = recent["accessionNumber"]
    primary_docs = recent["primaryDocument"]

    found = {}
    for form, rdate, acc, doc in zip(forms, report_dates, accessions, primary_docs):
        if form != "10-K":            # exact match: no amendments/quarterlies
            continue
        if not rdate:
            continue
        fy = int(rdate[:4])           # fiscal year from report date (YYYY-MM-DD)
        if fy in target_years and fy not in found:
            found[fy] = {"accession": acc, "primary_doc": doc, "report_date": rdate}
    return found

def download_filing(cik, accession, primary_doc):
    """Download the primary HTML document of one filing."""
    acc_nodash = accession.replace("-", "")
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_nodash}/{primary_doc}"
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.text

# ---- Main fetch loop ----
manifest = []
print("Fetching filings (this takes a few minutes; be patient & polite to SEC)\n")

for ticker, cik in BANKS.items():
    print(f"{ticker} (CIK {cik}):")
    try:
        subs = get_submissions(cik)
        time.sleep(0.3)                      # rate-limit courtesy
    except Exception as e:
        print(f"   [ERROR] could not fetch submissions: {e}")
        continue

    filings = find_10k_filings(subs, FISCAL_YEARS)

    for fy in FISCAL_YEARS:
        out_path = RAW_DIR / f"{ticker}_{fy}_10K.html"
        if out_path.exists():
            print(f"   [cached] FY{fy}")
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": str(out_path), "status": "cached"})
            continue
        if fy not in filings:
            print(f"   [gap]    no 10-K found for FY{fy}")
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": None, "status": "missing"})
            continue
        try:
            html = download_filing(cik, filings[fy]["accession"],
                                   filings[fy]["primary_doc"])
            out_path.write_text(html, encoding="utf-8")
            size = len(html.encode("utf-8"))
            print(f"   [saved]  FY{fy}  ({size:,} bytes)")
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": str(out_path), "status": "saved",
                             "bytes": size})
            time.sleep(0.3)                  # rate-limit courtesy
        except Exception as e:
            print(f"   [ERROR]  FY{fy}: {e}")
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": None, "status": f"error: {e}"})
    print()

# Save the manifest
manifest_path = PROJECT_ROOT / "data" / "processed" / "filings_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

saved  = sum(1 for m in manifest if m["status"] in ("saved", "cached"))
missing= sum(1 for m in manifest if m["status"] == "missing")
errors = sum(1 for m in manifest if str(m["status"]).startswith("error"))
print("="*55)
print(f"Done. saved/cached: {saved}   missing: {missing}   errors: {errors}")
print(f"Manifest: {manifest_path}")

Fetching filings (this takes a few minutes; be patient & polite to SEC)

SIVB (CIK 719739):
   [saved]  FY2016  (8,431,023 bytes)
   [saved]  FY2017  (8,582,832 bytes)
   [saved]  FY2018  (8,656,880 bytes)
   [saved]  FY2019  (11,581,682 bytes)
   [saved]  FY2020  (10,111,824 bytes)
   [saved]  FY2021  (9,702,463 bytes)
   [saved]  FY2022  (9,797,241 bytes)

PACW (CIK 1102112):
   [saved]  FY2016  (6,330,601 bytes)
   [saved]  FY2017  (6,165,529 bytes)
   [saved]  FY2018  (6,434,755 bytes)
   [saved]  FY2019  (8,035,662 bytes)
   [saved]  FY2020  (8,182,600 bytes)
   [saved]  FY2021  (9,095,266 bytes)
   [saved]  FY2022  (9,640,080 bytes)

WAL (CIK 1212545):
   [gap]    no 10-K found for FY2016
   [saved]  FY2017  (7,425,238 bytes)
   [saved]  FY2018  (7,551,194 bytes)
   [saved]  FY2019  (9,301,829 bytes)
   [saved]  FY2020  (7,565,796 bytes)
   [saved]  FY2021  (8,107,921 bytes)
   [saved]  FY2022  (8,389,799 bytes)

ZION (CIK 109380):
   [gap]    no 10-K found for FY2016
   [gap]   

In [ ]:
#why 14 missing? 
#Ans:-
#Now the gaps — and why they happened, because this is the important part. Those 14 [gap] results are not because the filings don't exist. 
#They exist on EDGAR. They're missing because of a limitation I deliberately left in the code to keep it simple: my find_10k_filings only reads 
#the recent block of EDGAR's submissions JSON, which holds a company's most recent ~1000 filings. For banks that file a lot, the older years 
#(2016–2018) get pushed into a separate "history" file that my code doesn't read yet. That's exactly why the gaps cluster in the early years and 
#why they hit different banks at different cutoffs (ZION and FITB file more, so their cutoff is more recent).

In [ ]:
#the fix done-
#fix the history-page fetch so you get the missing 2016–2018 years and have a fuller, more balanced panel. 
#It's about 15 lines of extra code (read the filings.files list in the submissions JSON, fetch each older-history JSON, merge). 
#More complete, and it removes a limitation an interviewer could poke at ("why is your panel unbalanced across years?").

In [7]:
# ==============================================================
# CELL 2c — Fetch EDGAR history pages & fill the gaps
# ==============================================================

def get_all_filings(cik):
    """
    Return a company's COMPLETE 10-K history by merging the 'recent'
    block with every archived history page EDGAR stores separately.
    """
    base = f"https://data.sec.gov/submissions/CIK{cik:010d}.json"
    r = requests.get(base, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    time.sleep(0.3)

    # Collect (form, reportDate, accession, primaryDocument) from every source.
    records = []

    def add_block(block):
        forms = block.get("form", [])
        rdates = block.get("reportDate", [])
        accs  = block.get("accessionNumber", [])
        docs  = block.get("primaryDocument", [])
        for form, rdate, acc, doc in zip(forms, rdates, accs, docs):
            records.append((form, rdate, acc, doc))

    # 1) the recent block
    add_block(data["filings"]["recent"])

    # 2) each archived history page
    for extra in data["filings"].get("files", []):
        name = extra["name"]                       # e.g. CIK...-submissions-001.json
        url  = f"https://data.sec.gov/submissions/{name}"
        try:
            rr = requests.get(url, headers=HEADERS, timeout=30)
            rr.raise_for_status()
            add_block(rr.json())
            time.sleep(0.3)
        except Exception as e:
            print(f"      [warn] history page {name} failed: {e}")

    return records

def pick_10ks(records, target_years):
    """One 10-K per target fiscal year, from the merged record list."""
    found = {}
    for form, rdate, acc, doc in records:
        if form != "10-K" or not rdate:
            continue
        fy = int(rdate[:4])
        if fy in target_years and fy not in found:
            found[fy] = {"accession": acc, "primary_doc": doc, "report_date": rdate}
    return found

# ---- Re-run, filling only the gaps ----
print("Filling gaps using full EDGAR history\n")
filled = 0

for ticker, cik in BANKS.items():
    # Which years are still missing on disk?
    missing_years = [fy for fy in FISCAL_YEARS
                     if not (RAW_DIR / f"{ticker}_{fy}_10K.html").exists()]
    if not missing_years:
        print(f"{ticker}: complete, nothing to fill")
        continue

    print(f"{ticker}: missing {missing_years} -> searching full history")
    try:
        all_recs = get_all_filings(cik)
        filings  = pick_10ks(all_recs, FISCAL_YEARS)
    except Exception as e:
        print(f"   [ERROR] {e}")
        continue

    for fy in missing_years:
        if fy not in filings:
            print(f"   [still-gap] FY{fy} genuinely not on EDGAR")
            continue
        try:
            html = download_filing(cik, filings[fy]["accession"],
                                   filings[fy]["primary_doc"])
            out_path = RAW_DIR / f"{ticker}_{fy}_10K.html"
            out_path.write_text(html, encoding="utf-8")
            size = len(html.encode("utf-8"))
            print(f"   [saved]  FY{fy}  ({size:,} bytes)")
            filled += 1
            time.sleep(0.3)
        except Exception as e:
            print(f"   [ERROR]  FY{fy}: {e}")
    print()

print("="*55)
print(f"Gap-fill complete. Newly downloaded: {filled}")

# Rebuild the manifest from what's actually on disk now
manifest = []
for ticker in BANKS:
    for fy in FISCAL_YEARS:
        p = RAW_DIR / f"{ticker}_{fy}_10K.html"
        if p.exists():
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": str(p), "status": "present",
                             "bytes": p.stat().st_size})
        else:
            manifest.append({"ticker": ticker, "fiscal_year": fy,
                             "path": None, "status": "missing"})

manifest_path = PROJECT_ROOT / "data" / "processed" / "filings_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

present = sum(1 for m in manifest if m["status"] == "present")
print(f"Corpus now: {present}/{len(BANKS)*len(FISCAL_YEARS)} filings present")
print(f"Manifest updated: {manifest_path}")

Filling gaps using full EDGAR history

SIVB: complete, nothing to fill
PACW: complete, nothing to fill
WAL: missing [2016] -> searching full history
   [saved]  FY2016  (7,753,153 bytes)

ZION: missing [2016, 2017, 2018, 2019] -> searching full history
   [saved]  FY2016  (7,703,947 bytes)
   [saved]  FY2017  (7,170,315 bytes)
   [saved]  FY2018  (7,207,076 bytes)
   [saved]  FY2019  (11,458,102 bytes)

CMA: missing [2016, 2017] -> searching full history
   [saved]  FY2016  (7,323,980 bytes)
   [saved]  FY2017  (7,195,172 bytes)

KEY: missing [2016, 2017] -> searching full history
   [saved]  FY2016  (9,124,093 bytes)
   [saved]  FY2017  (8,268,296 bytes)

RF: missing [2016, 2017] -> searching full history
   [saved]  FY2016  (8,014,881 bytes)
   [saved]  FY2017  (7,722,004 bytes)

FITB: missing [2016, 2017, 2018] -> searching full history
   [saved]  FY2016  (4,541,046 bytes)
   [saved]  FY2017  (4,600,544 bytes)
   [saved]  FY2018  (4,710,493 bytes)

Gap-fill complete. Newly download

In [ ]:
# The problem: A 10-K is a huge HTML document (millions of bytes) covering the whole annual report. We only want Item 1A, "Risk Factors" — 
# the section where management discloses what could go wrong. It sits between the "Item 1A" header and the next header ("Item 1B" or "Item 2"). 
# But the HTML is messy: headers appear in tables of contents and in the body, formatting varies by company and year, and some filings say 
# "Risk Factors" in a dozen incidental places. Naive extraction grabs the wrong span or returns near-nothing — which is exactly how HBAN silently 
# became "1 word."

# The strategy: strip HTML to clean text, then find the real Item 1A section (not its table-of-contents mention) by locating the header and reading 
# to the next item header. Then — critically — audit every extraction so a failure can't flow downstream unseen.

In [8]:
# ==============================================================
# CELL 3a — HTML to clean text (tested on ONE file first)
# ==============================================================
from bs4 import BeautifulSoup
import re

def html_to_text(html):
    """Strip a 10-K's HTML down to readable plain text."""
    soup = BeautifulSoup(html, "lxml")
    # Remove script/style noise
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    # Normalise whitespace: collapse runs of spaces/newlines
    text = re.sub(r"\s+", " ", text)
    # Fix non-breaking spaces and common artifacts
    text = text.replace("\xa0", " ")
    return text.strip()

# --- Test on a single SIVB filing before batch-processing ---
test_path = RAW_DIR / "SIVB_2022_10K.html"
html = test_path.read_text(encoding="utf-8")
text = html_to_text(html)

print(f"Test file: {test_path.name}")
print(f"Raw HTML length : {len(html):,} chars")
print(f"Clean text length: {len(text):,} chars")
print(f"Word count       : {len(text.split()):,} words")
print("\n--- First 500 chars of clean text ---")
print(text[:500])
print("\n--- Does 'Item 1A' appear? ---")
# Show where 'item 1a' occurs (case-insensitive)
positions = [m.start() for m in re.finditer(r"item\s*1a", text, re.IGNORECASE)]
print(f"'Item 1A' appears {len(positions)} times, at positions: {positions[:10]}")

C:\Users\aradh\AppData\Local\Temp\ipykernel_18844\2254769158.py:9: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Test file: SIVB_2022_10K.html
Raw HTML length : 9,797,241 chars
Clean text length: 840,963 chars
Word count       : 115,588 words

--- First 500 chars of clean text ---
sivb-20221231 false 2022 FY 0000719739 P3M P3D P3D P3D P3D 0.025 0.01 0.01 0.01 0.01 0.025 0.01 0.01 0.01 0.01 http://fasb.org/us-gaap/2022#OtherAssets http://fasb.org/us-gaap/2022#OtherAssets http://fasb.org/us-gaap/2022#OtherLiabilities http://fasb.org/us-gaap/2022#OtherLiabilities P3M 0000719739 2022-01-01 2022-12-31 0000719739 us-gaap:CommonStockMember 2022-01-01 2022-12-31 0000719739 us-gaap:SeriesAPreferredStockMember 2022-01-01 2022-12-31 0000719739 2022-06-30 iso4217:USD 0000719739 2023-

--- Does 'Item 1A' appear? ---
'Item 1A' appears 5 times, at positions: [151323, 162457, 228189, 359215, 469585]


In [9]:
# 151348 and 151386 are character positions — how many characters into the cleaned text that phrase appears.
# checked what and where is the actual Iteam 1A for one bank- but it has occured 5 times. possibly the first time will be in table of contents, 
# second one - the actual header and then later references- like, from section 1A we have...

#so, let us check what is what actually

In [10]:
# ==============================================================
# CELL 3b — Inspect each "Item 1A" occurrence (diagnostic only)
# ==============================================================
# Reuse `text` from cell 3a (SIVB 2022 clean text).
import re

positions = [m.start() for m in re.finditer(r"item\s*1a", text, re.IGNORECASE)]

print(f"'Item 1A' appears {len(positions)} times.\n")
for i, pos in enumerate(positions):
    # Show 120 chars of context around each hit
    snippet = text[pos : pos + 120].replace("\n", " ")
    print(f"[{i}] position {pos:>8}:  ...{snippet}...")
    print()

# Also locate the candidate END headers (Item 1B / Item 2)
print("="*60)
print("Candidate END markers:")
for pat_name, pat in [("Item 1B", r"item\s*1b"), ("Item 2", r"item\s*2[^0-9]")]:
    hits = [m.start() for m in re.finditer(pat, text, re.IGNORECASE)]
    print(f"  {pat_name}: {len(hits)} occurrences at {hits[:8]}")

'Item 1A' appears 5 times.

[0] position   151323:  ...Item 1A. Risk Factors 17 Item 1B. Unresolved Staff Comments 37 Item 2. Properties 37 Item 3. Legal Proceedings 38 Item 4...

[1] position   162457:  ...Item 1A of this report. Accordingly, you are cautioned not to place undue reliance on forward-looking statements. We urg...

[2] position   228189:  ...ITEM 1A. RISK FACTORS Our business faces material risks, including credit, market and liquidity, operational, legal and ...

[3] position   359215:  ...Item 1A of this report. Our fiscal year ends December 31 st and, unless otherwise noted, references to years or fiscal y...

[4] position   469585:  ...Item 1A of this report. 72 Table of Contents As of December 31, 2022, 92 percent, or $68.4 billion, of our outstanding t...

Candidate END markers:
  Item 1B: 2 occurrences at [151348, 352066]
  Item 2: 2 occurrences at [151386, 352107]


In [12]:
#seeing the output, 228189 is the real Iteam1A.
#So the real Risk Factors section runs from 228189 (the real ITEM 1A. RISK FACTORS header) to 352066 (the real Item 1B).

In [13]:
#Here's the key insight this gives us for the extractor, and it's the interview-worthy part: the real section header tends to be the ALL-CAPS 
#standalone version (ITEM 1A. RISK FACTORS), while cross-references say Item 1A of this report in mixed case mid-sentence. And the real section 
#is the longest span between an Item 1A header and the following Item 1B/2 header. We'll use both signals — pick the Item 1A occurrence that 
#(a) is followed closely by "Risk Factors" as a header and (b) yields the longest, most content-rich span to the next item marker.

In [15]:
# ==============================================================
# CELL 3c (v2) — Item 1A extractor, fixed TOC rejection
# ==============================================================
def extract_item1a(text):
    """
    Extract the real Item 1A (Risk Factors) section.

    Key fix: distinguish the REAL section header from the table-of-contents
    line. In the TOC, 'Item 1A Risk Factors' is followed by a PAGE NUMBER
    then the next item ('...Risk Factors 17 Item 1B...'). In the real
    section, it's followed by PROSE ('...Risk Factors Our business faces...').
    We detect prose vs a page-number/next-item pattern right after the header.
    """
    candidates = []
    for m in re.finditer(r"item\s*1a[.\s]*risk\s*factors", text, re.IGNORECASE):
        s = m.start()
        after = text[m.end() : m.end() + 60].strip()
        # TOC tell: begins with a page number and/or quickly hits 'Item 1B'
        looks_like_toc = bool(re.match(r"^\d{1,3}\b", after)) or \
                         bool(re.search(r"item\s*1b", after, re.IGNORECASE))
        candidates.append({"start": s, "after": after, "is_toc": looks_like_toc})

    # Prefer non-TOC candidates (the real section header)
    real = [c for c in candidates if not c["is_toc"]]
    pool = real if real else candidates  # fallback: use all if none pass

    if not pool:
        return {"text": "", "n_words": 0, "status": "no_section_found"}

    # End markers: real Item 1B / Item 2 headers
    end_markers = sorted(
        [m.start() for m in re.finditer(r"item\s*1b", text, re.IGNORECASE)] +
        [m.start() for m in re.finditer(r"item\s*2[^0-9a-z]", text, re.IGNORECASE)]
    )

    # For each real start, take the NEAREST end marker after it; pick the
    # candidate giving the most plausible section (longest among real starts).
    best = None
    for c in pool:
        s = c["start"]
        later = [e for e in end_markers if e > s + 200]  # must be well after
        if not later:
            continue
        e = min(later)
        span = text[s:e]
        n_words = len(span.split())
        if best is None or n_words > best["n_words"]:
            best = {"text": span, "n_words": n_words, "start": s, "end": e}

    if best is None:
        return {"text": "", "n_words": 0, "status": "no_end_marker"}

    n = best["n_words"]
    status = "too_short_SUSPECT" if n < 500 else "short_check" if n < 3000 else "ok"
    best["status"] = status
    return best

# --- Re-test on SIVB 2022 ---
result = extract_item1a(text)
print("SIVB 2022 extraction (v2):")
print(f"  status : {result['status']}")
print(f"  words  : {result['n_words']:,}")
print(f"  span   : chars {result.get('start')} -> {result.get('end')}")
print(f"\n  First 300 chars:")
print("  " + result["text"][:300])

SIVB 2022 extraction (v2):
  status : ok
  words  : 18,638
  span   : chars 228189 -> 352066

  First 300 chars:
  ITEM 1A. RISK FACTORS Our business faces material risks, including credit, market and liquidity, operational, legal and regulatory and strategic and reputational risks. The factors described below are not intended to serve as a comprehensive listing of the risks we face. Additional risks and uncerta


In [16]:
#now we have checked, audited and found the risk section for SIVB. Now, we will do the same for all 56 filings

In [17]:
# ==============================================================
# CELL 3d — Batch extract Item 1A across all 56 filings + AUDIT
# ==============================================================
import pandas as pd
import json

PROCESSED = PROJECT_ROOT / "data" / "processed"
ITEM1A_DIR = PROCESSED / "item1a"
ITEM1A_DIR.mkdir(parents=True, exist_ok=True)

records = []
print("Extracting Item 1A from all filings\n")

for ticker in BANKS:
    for fy in FISCAL_YEARS:
        raw_path = RAW_DIR / f"{ticker}_{fy}_10K.html"
        if not raw_path.exists():
            records.append({"ticker": ticker, "fiscal_year": fy,
                            "n_words": 0, "status": "file_missing"})
            continue
        html = raw_path.read_text(encoding="utf-8")
        clean = html_to_text(html)
        res = extract_item1a(clean)

        # Save the extracted section text to its own file
        if res["n_words"] > 0:
            out_txt = ITEM1A_DIR / f"{ticker}_{fy}_item1a.txt"
            out_txt.write_text(res["text"], encoding="utf-8")

        records.append({"ticker": ticker, "fiscal_year": fy,
                        "n_words": res["n_words"], "status": res["status"]})

df = pd.DataFrame(records)

# --- AUDIT TABLE ---
print("="*60)
print("EXTRACTION AUDIT")
print("="*60)

# 1. Status breakdown
print("\nStatus counts:")
print(df["status"].value_counts().to_string())

# 2. Coverage grid of word counts (rows=ticker, cols=year)
print("\nWord-count grid (rows=bank, cols=fiscal year):")
grid = df.pivot(index="ticker", columns="fiscal_year", values="n_words")
print(grid.to_string())

# 3. Flag suspects: anything under 3000 words or way off a bank's own median
print("\n" + "-"*60)
print("SUSPECT extractions (word count < 3000 OR < 40% of bank median):")
df["bank_median"] = df.groupby("ticker")["n_words"].transform("median")
df["pct_of_median"] = (df["n_words"] / df["bank_median"] * 100).round(1)
suspects = df[(df["n_words"] < 3000) | (df["pct_of_median"] < 40)]
if len(suspects):
    print(suspects[["ticker","fiscal_year","n_words","pct_of_median","status"]].to_string(index=False))
else:
    print("  none — all extractions look healthy")

# Save the manifest
df.to_csv(PROCESSED / "item1a_manifest.csv", index=False)
print(f"\nSaved: {PROCESSED / 'item1a_manifest.csv'}")
print(f"Extracted text files in: {ITEM1A_DIR}")

Extracting Item 1A from all filings



C:\Users\aradh\AppData\Local\Temp\ipykernel_18844\2254769158.py:9: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


EXTRACTION AUDIT

Status counts:
status
ok    56

Word-count grid (rows=bank, cols=fiscal year):
fiscal_year   2016   2017   2018   2019   2020   2021   2022
ticker                                                      
CMA           5665   6590   8070   7758   8595   8323   8551
FITB         11167  11364  12475  11999  13475  16443  15841
KEY           7343   7169   7615   8048   9384   9519  10414
PACW         20391  21270  21110  21634  24873  25686  25689
RF           12645  12141  12836  13252  13911  14857  17495
SIVB         12561  12526  13135  13430  16926  18036  18638
WAL           7846   8468   8956   9217  11104  11503  11032
ZION          5685   5643   6699   6577   7493   5667   5927

------------------------------------------------------------
SUSPECT extractions (word count < 3000 OR < 40% of bank median):
  none — all extractions look healthy

Saved: E:\project_placements\unsupervised\data\processed\item1a_manifest.csv
Extracted text files in: E:\project_placements\uns

In [ ]:
#SIVB shows a real upward trend — 12,561 words (2016) climbing to 18,638 (2022). Its Risk Factors section grew ~48% over the period. 
# That's a genuine signal worth examining later: did SVB's disclosure length expand as its balance sheet risk grew? (Length is one 
# of your features — this is the kind of thing the analysis will quantify properly rather than eyeball.)

# PACW is an outlier in length — 20K–25K words, roughly double the others. Not a bug (it's consistent across all years and the text is real), 
# just a bank that writes verbose risk sections. Worth noting because it'll affect any raw-length comparison; your features should normalize for 
# this where appropriate.

# ZION is consistently the shortest (~5,600–7,500). Again consistent, so real, not a parse failure.

# Preprocessing

In [18]:
# ==============================================================
# CELL 4 — Preprocessing: build TWO cleaned versions per document
# ==============================================================
import re
import nltk
import pandas as pd
from pathlib import Path
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# One-time NLTK data (safe to re-run)
for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

PROCESSED  = PROJECT_ROOT / "data" / "processed"
ITEM1A_DIR = PROCESSED / "item1a"

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_light(text):
    """
    LIGHT cleaning for readability & tone features.
    Keeps sentences, case, and punctuation. Removes XBRL/boilerplate noise
    and normalises whitespace only.
    """
    # Remove obvious XBRL/URL artifacts (fasb.org tags, us-gaap:, iso4217, CIK strings)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"\b(us-gaap|iso4217|xbrli|dei):\S+", " ", text)
    text = re.sub(r"\b\d{10}\b", " ", text)          # 10-digit CIK codes
    # Normalise whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_heavy(text):
    """
    HEAVY cleaning for TF-IDF, semantic drift, and LDA.
    Lowercase -> tokenize -> keep alphabetic -> drop stopwords -> lemmatize.
    """
    text = text.lower()
    tokens = word_tokenize(text)
    out = []
    for tok in tokens:
        if not tok.isalpha():        # drop numbers, punctuation
            continue
        if tok in STOPWORDS:         # drop common stopwords
            continue
        if len(tok) < 3:             # drop very short tokens
            continue
        out.append(lemmatizer.lemmatize(tok))
    return " ".join(out)

# --- Process all 56 sections ---
rows = []
print("Preprocessing 56 sections into light + heavy versions\n")

for txt_path in sorted(ITEM1A_DIR.glob("*_item1a.txt")):
    name = txt_path.stem                      # e.g. SIVB_2022_item1a
    ticker, fy, _ = name.split("_")
    raw = txt_path.read_text(encoding="utf-8")

    light = clean_light(raw)
    heavy = clean_heavy(raw)

    rows.append({
        "ticker": ticker,
        "fiscal_year": int(fy),
        "text_light": light,     # for readability + tone
        "text_heavy": heavy,     # for tf-idf, sbert, lda
        "n_words_light": len(light.split()),
        "n_words_heavy": len(heavy.split()),
    })

corpus = pd.DataFrame(rows).sort_values(["ticker", "fiscal_year"]).reset_index(drop=True)

# Save the preprocessed corpus (pickle preserves the long text cleanly)
corpus.to_pickle(PROCESSED / "corpus_preprocessed.pkl")

print("="*60)
print(f"Preprocessed {len(corpus)} documents")
print("\nSample (word counts before/after heavy cleaning):")
print(corpus[["ticker","fiscal_year","n_words_light","n_words_heavy"]].head(10).to_string(index=False))
print(f"\nHeavy cleaning removed ~{100*(1 - corpus['n_words_heavy'].sum()/corpus['n_words_light'].sum()):.0f}% of tokens (stopwords/numbers/short) — expected.")
print(f"\nSaved: {PROCESSED / 'corpus_preprocessed.pkl'}")

Preprocessing 56 sections into light + heavy versions

Preprocessed 56 documents

Sample (word counts before/after heavy cleaning):
ticker  fiscal_year  n_words_light  n_words_heavy
   CMA         2016           5665           3492
   CMA         2017           6590           4035
   CMA         2018           8070           4968
   CMA         2019           7758           4796
   CMA         2020           8595           5288
   CMA         2021           8323           5142
   CMA         2022           8551           5313
  FITB         2016          11167           6712
  FITB         2017          11364           6885
  FITB         2018          12475           7467

Heavy cleaning removed ~42% of tokens (stopwords/numbers/short) — expected.

Saved: E:\project_placements\unsupervised\data\processed\corpus_preprocessed.pkl


In [20]:
# ==============================================================
# CELL 5a — Load the Loughran-McDonald master dictionary
# ==============================================================
import pandas as pd
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
LM_DIR = PROJECT_ROOT / "data" / "lexicons"
LM_DIR.mkdir(parents=True, exist_ok=True)

# The master dictionary CSV. Download it once from:
#   https://sraf.nd.edu/loughranmcdonald-master-dictionary/
# Get the file named like "Loughran-McDonald_MasterDictionary_1993-2024.csv"
# (a few MB — NOT the 16 GB DocumentDictionaries file).
# Save it into: data/lexicons/
LM_CSV = LM_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv"

if not LM_CSV.exists():
    print("!!! LM dictionary not found.")
    print("Download it manually (one time):")
    print("  1. Go to https://sraf.nd.edu/loughranmcdonald-master-dictionary/")
    print("  2. Download the Master Dictionary CSV (a few MB, 1993-2024).")
    print(f"  3. Save it as:\n     {LM_CSV}")
    print("\nThen re-run this cell.")
else:
    lm = pd.read_csv(LM_CSV)
    print(f"Loaded LM dictionary: {lm.shape[0]:,} words, {lm.shape[1]} columns")
    print("\nColumns:", list(lm.columns))
    # The sentiment columns hold a YEAR (first year the word entered that
    # category) or 0 if the word is NOT in that category.
    print("\nSample rows:")
    print(lm.head(3).to_string())

Loaded LM dictionary: 86,553 words, 17 columns

Columns: ['Word', 'Seq_num', 'Word Count', 'Word Proportion', 'Average Proportion', 'Std Dev', 'Doc Count', 'Negative', 'Positive', 'Uncertainty', 'Litigious', 'Strong_Modal', 'Weak_Modal', 'Constraining', 'Complexity', 'Syllables', 'Source']

Sample rows:
        Word  Seq_num  Word Count  Word Proportion  Average Proportion       Std Dev  Doc Count  Negative  Positive  Uncertainty  Litigious  Strong_Modal  Weak_Modal  Constraining  Complexity  Syllables     Source
0   AARDVARK        1         814     3.085383e-08        2.022181e-08  4.059339e-06        158         0         0            0          0             0           0             0           0          2  12of12inf
1  AARDVARKS        2           3     1.137119e-10        7.898767e-12  8.829342e-09          1         0         0            0          0             0           0             0           0          2  12of12inf
2      ABACI        3           9     3.411357e-10   

# LM tone scoring

In [21]:
# ==============================================================
# CELL 5b — Loughran-McDonald tone scoring (all 56 filings)
# ==============================================================
import pandas as pd
import re
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"

# --- Build category -> set-of-words lookups from the LM dictionary ---
# A word is IN a category if that column value is a POSITIVE year (>0).
SENTIMENT_COLS = ["Negative", "Positive", "Uncertainty",
                  "Litigious", "Strong_Modal", "Weak_Modal", "Constraining"]

lm_sets = {}
for col in SENTIMENT_COLS:
    words_in_cat = set(lm.loc[lm[col] > 0, "Word"].str.upper())
    lm_sets[col] = words_in_cat
    print(f"  {col:14s}: {len(words_in_cat):>5,} words")

# --- Load the preprocessed corpus (light text = for tone) ---
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")

def score_tone(text_light):
    """
    Count LM category words as a proportion of total words.
    Uses the light-cleaned text (stopwords retained) so the denominator
    is the true word count, matching the standard LM methodology.
    """
    # Tokenise to uppercase alphabetic words (LM dictionary is uppercase)
    words = re.findall(r"[A-Za-z]+", text_light.upper())
    total = len(words)
    if total == 0:
        return {f"{c.lower()}_pct": 0.0 for c in SENTIMENT_COLS} | {"total_words": 0}

    counts = {c: 0 for c in SENTIMENT_COLS}
    for w in words:
        for c in SENTIMENT_COLS:
            if w in lm_sets[c]:
                counts[c] += 1

    result = {f"{c.lower()}_pct": round(100 * counts[c] / total, 4)
              for c in SENTIMENT_COLS}
    result["total_words"] = total
    return result

# --- Apply to all documents ---
print("\nScoring tone for all 56 filings...")
tone_rows = []
for _, row in corpus.iterrows():
    scores = score_tone(row["text_light"])
    tone_rows.append({"ticker": row["ticker"],
                      "fiscal_year": row["fiscal_year"],
                      **scores})

tone = pd.DataFrame(tone_rows)
tone.to_csv(PROCESSED / "features_tone.csv", index=False)

# --- Show SVB's tone trajectory (the headline for the longitudinal test) ---
print("\n" + "="*60)
print("SVB (SIVB) tone over time — the key trajectory:")
print("="*60)
svb = tone[tone["ticker"] == "SIVB"].sort_values("fiscal_year")
print(svb[["fiscal_year","negative_pct","uncertainty_pct",
           "litigious_pct","total_words"]].to_string(index=False))

print("\n2022 peer snapshot — negative tone, all banks:")
snap = tone[tone["fiscal_year"] == 2022].sort_values("negative_pct", ascending=False)
print(snap[["ticker","negative_pct","uncertainty_pct","litigious_pct"]].to_string(index=False))

print(f"\nSaved: {PROCESSED / 'features_tone.csv'}")

  Negative      : 2,345 words
  Positive      :   347 words
  Uncertainty   :   297 words
  Litigious     :   903 words
  Strong_Modal  :    19 words
  Weak_Modal    :    27 words
  Constraining  :   184 words

Scoring tone for all 56 filings...

SVB (SIVB) tone over time — the key trajectory:
 fiscal_year  negative_pct  uncertainty_pct  litigious_pct  total_words
        2016        3.9566           3.4264         1.5273        12637
        2017        4.0821           3.4401         1.4902        12616
        2018        4.0176           3.3669         1.5359        13217
        2019        4.1602           3.3519         1.4980        13485
        2020        4.3187           3.5714         1.4003        16996
        2021        4.2257           3.5913         1.3626        18127
        2022        4.3223           3.6126         1.3607        18740

2022 peer snapshot — negative tone, all banks:
ticker  negative_pct  uncertainty_pct  litigious_pct
   KEY        4.7006        

# Readability

In [22]:
# ==============================================================
# CELL 6 — Readability (Gunning Fog index), all 56 filings
# ==============================================================
import pandas as pd
import re
from pathlib import Path
import nltk
from nltk.tokenize import sent_tokenize

PROCESSED = PROJECT_ROOT / "data" / "processed"

# --- Syllable lookup from the LM dictionary (reuse, no new library) ---
# lm is still in memory from cell 5a. Build WORD -> syllable-count map.
syllable_map = dict(zip(lm["Word"].str.upper(), lm["Syllables"]))

def count_syllables(word):
    """Syllables from LM dict; fallback = vowel-group heuristic."""
    w = word.upper()
    if w in syllable_map and syllable_map[w] > 0:
        return int(syllable_map[w])
    # Fallback: count vowel groups (rough but fine for rare words)
    groups = re.findall(r"[AEIOUY]+", w)
    return max(1, len(groups))

def fog_index(text_light):
    """Gunning Fog on sentence-intact (light) text."""
    sentences = sent_tokenize(text_light)
    n_sentences = len([s for s in sentences if s.strip()])
    words = re.findall(r"[A-Za-z]+", text_light)
    n_words = len(words)
    if n_sentences == 0 or n_words == 0:
        return {"fog": 0.0, "words_per_sentence": 0.0,
                "pct_complex": 0.0, "n_sentences": 0}

    complex_words = sum(1 for w in words if count_syllables(w) >= 3)
    wps = n_words / n_sentences
    pct_complex = 100 * complex_words / n_words
    fog = 0.4 * (wps + pct_complex)
    return {"fog": round(fog, 3),
            "words_per_sentence": round(wps, 2),
            "pct_complex": round(pct_complex, 2),
            "n_sentences": n_sentences}

# --- Apply to all documents ---
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")
print("Computing Fog readability for all 56 filings...\n")

read_rows = []
for _, row in corpus.iterrows():
    r = fog_index(row["text_light"])
    read_rows.append({"ticker": row["ticker"],
                      "fiscal_year": row["fiscal_year"], **r})

readability = pd.DataFrame(read_rows)
readability.to_csv(PROCESSED / "features_readability.csv", index=False)

# --- SVB trajectory + 2022 peer snapshot ---
print("="*60)
print("SVB (SIVB) readability over time:")
print("="*60)
svb = readability[readability["ticker"] == "SIVB"].sort_values("fiscal_year")
print(svb[["fiscal_year","fog","words_per_sentence","pct_complex"]].to_string(index=False))

print("\n2022 peer snapshot — Fog index (higher = denser/harder):")
snap = readability[readability["fiscal_year"] == 2022].sort_values("fog", ascending=False)
print(snap[["ticker","fog","words_per_sentence","pct_complex"]].to_string(index=False))

print(f"\nSaved: {PROCESSED / 'features_readability.csv'}")

Computing Fog readability for all 56 filings...

SVB (SIVB) readability over time:
 fiscal_year    fog  words_per_sentence  pct_complex
        2016 23.888               33.79        25.93
        2017 24.043               34.28        25.82
        2018 24.248               34.51        26.11
        2019 24.146               34.23        26.14
        2020 24.150               34.69        25.69
        2021 24.110               34.33        25.94
        2022 24.012               33.89        26.14

2022 peer snapshot — Fog index (higher = denser/harder):
ticker    fog  words_per_sentence  pct_complex
   CMA 24.154               29.11        31.27
  SIVB 24.012               33.89        26.14
  PACW 23.632               31.83        27.25
    RF 23.588               31.79        27.18
   WAL 23.092               31.72        26.01
  FITB 22.949               31.76        25.61
   KEY 22.229               28.66        26.92
  ZION 22.208               27.21        28.31

Saved: E:\p

# N-gram / key-phrase analysis

In [23]:
# ==============================================================
# CELL 7 — N-gram analysis + risk-term frequency tracking
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")

# ---------------------------------------------------------------
# PART 1 — Top bigrams & trigrams across the whole corpus
# Uses HEAVY text (stopwords already removed) so phrases are contentful.
# ---------------------------------------------------------------
def top_ngrams(texts, ngram_range, top_k=15):
    vec = CountVectorizer(ngram_range=ngram_range, min_df=3)
    X = vec.fit_transform(texts)
    freqs = np.asarray(X.sum(axis=0)).ravel()
    vocab = np.array(vec.get_feature_names_out())
    order = freqs.argsort()[::-1][:top_k]
    return list(zip(vocab[order], freqs[order]))

print("="*60)
print("TOP BIGRAMS (whole corpus):")
print("="*60)
for phrase, cnt in top_ngrams(corpus["text_heavy"], (2, 2), 15):
    print(f"  {cnt:>6,}  {phrase}")

print("\n" + "="*60)
print("TOP TRIGRAMS (whole corpus):")
print("="*60)
for phrase, cnt in top_ngrams(corpus["text_heavy"], (3, 3), 15):
    print(f"  {cnt:>6,}  {phrase}")

# ---------------------------------------------------------------
# PART 2 — Track SVB-relevant risk terms over time (rate per 1,000 words)
# These are the risks that actually caused the collapse.
# Search in LIGHT text (real words, real phrasing preserved).
# ---------------------------------------------------------------
RISK_TERMS = [
    "interest rate", "liquidity", "deposit", "unrealized loss",
    "available for sale", "held to maturity", "capital",
    "concentration", "uninsured", "duration",
]

def term_rate(text_light, term):
    """Occurrences of `term` per 1,000 words."""
    t = text_light.lower()
    n_words = len(t.split())
    if n_words == 0:
        return 0.0
    count = t.count(term.lower())
    return round(1000 * count / n_words, 3)

print("\n" + "="*60)
print("SVB risk-term frequency over time (per 1,000 words):")
print("="*60)
svb = corpus[corpus["ticker"] == "SIVB"].sort_values("fiscal_year")
rows = []
for _, r in svb.iterrows():
    row = {"fiscal_year": r["fiscal_year"]}
    for term in RISK_TERMS:
        row[term] = term_rate(r["text_light"], term)
    rows.append(row)
svb_terms = pd.DataFrame(rows)
pd.set_option("display.max_columns", None, "display.width", 140)
print(svb_terms.to_string(index=False))

svb_terms.to_csv(PROCESSED / "features_svb_riskterms.csv", index=False)
print(f"\nSaved: {PROCESSED / 'features_svb_riskterms.csv'}")
print("\nRead the trend: did 'interest rate', 'liquidity', 'deposit',")
print("'unrealized loss', 'uninsured' RISE toward 2022 — or stay flat?")

TOP BIGRAMS (whole corpus):
   2,345  fifth third
   1,658  adversely affect
   1,605  result operation
   1,581  financial condition
   1,072  interest rate
   1,026  could adversely
     958  real estate
     952  condition result
     768  financial service
     765  adverse effect
     762  financial institution
     721  third party
     705  product service
     667  credit loss
     650  business financial

TOP TRIGRAMS (whole corpus):
     952  financial condition result
     940  condition result operation
     769  could adversely affect
     553  business financial condition
     488  material adverse effect
     426  financial service industry
     419  could material adverse
     368  may adversely affect
     367  adversely affect business
     353  business result operation
     351  adverse effect business
     351  result operation financial
     349  operation financial condition
     276  could materially adversely
     263  fifth third may

SVB risk-term frequency o

In [ ]:
# Two honest caveats to keep ready, because a sharp interviewer will probe:

#Keyword counting is crude — "unrealized loss" as an exact bigram misses "unrealized losses" (plural) or "losses that are unrealized." 
# Your .count() on exact phrases undercounts. Before you lean hard on the zero, you'd want to confirm with a slightly looser match 
# (e.g. "unrealized" alone). Worth checking so you don't overclaim a literal zero that's partly a matching artifact.
#This is descriptive, not causal — you're showing the language didn't foreground these risks, not proving disclosure should have caught them. 
#The honest framing is "the words didn't track the danger," not "SVB hid it" (that's a legal claim you can't make from word counts).

#Given the first caveat, I'd suggest one quick robustness check next: re-run "unrealized," "uninsured," and "duration" as single-word rates 
#(looser matching) to confirm the near-zero holds and isn't just an exact-phrase artifact. It's a 5-line addition and it protects your headline finding.

# Robustness check with looser matching

In [24]:
# ==============================================================
# CELL 7b — Robustness: single-word (looser) matching
# ==============================================================
import pandas as pd
import re
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")

# Single-word / stem terms — looser than the exact phrases in 7.
# Using word-boundary regex so 'insured' doesn't match inside other words,
# and a stem for unrealiz(e/ed) to catch both spellings.
LOOSE_TERMS = {
    "unrealized":   r"\bunrealiz",       # unrealized / unrealised
    "uninsured":    r"\buninsured\b",
    "insured":      r"\binsured\b",       # for contrast
    "duration":     r"\bduration\b",
    "held-to-maturity": r"held[\s-]*to[\s-]*maturity",
    "available-for-sale": r"available[\s-]*for[\s-]*sale",
    "interest rate": r"\binterest\s+rate",
    "liquidity":    r"\bliquidit",        # liquidity / liquid
    "deposit":      r"\bdeposit",         # deposit / deposits / depositor
}

def loose_rate(text_light, pattern):
    """Regex matches per 1,000 words (case-insensitive)."""
    n_words = len(text_light.split())
    if n_words == 0:
        return 0.0
    hits = len(re.findall(pattern, text_light, re.IGNORECASE))
    return round(1000 * hits / n_words, 3)

svb = corpus[corpus["ticker"] == "SIVB"].sort_values("fiscal_year")
rows = []
for _, r in svb.iterrows():
    row = {"fiscal_year": r["fiscal_year"]}
    for label, pat in LOOSE_TERMS.items():
        row[label] = loose_rate(r["text_light"], pat)
    rows.append(row)

loose = pd.DataFrame(rows)
pd.set_option("display.max_columns", None, "display.width", 160)
print("="*70)
print("SVB risk-term frequency — LOOSE single-word matching (per 1,000 words):")
print("="*70)
print(loose.to_string(index=False))

print("\n--- Compare the critical terms to the exact-phrase version ---")
print("If 'unrealized' and 'uninsured' are STILL near-zero and flat,")
print("the headline finding holds. If they jumped, we revise it.")

SVB risk-term frequency — LOOSE single-word matching (per 1,000 words):
 fiscal_year  unrealized  uninsured  insured  duration  held-to-maturity  available-for-sale  interest rate  liquidity  deposit
        2016       0.159      0.080      0.0     0.080               0.0                 0.0          2.229      1.831    0.717
        2017       0.160      0.080      0.0     0.080               0.0                 0.0          2.235      1.916    0.719
        2018       0.152      0.076      0.0     0.076               0.0                 0.0          2.208      2.056    0.837
        2019       0.149      0.074      0.0     0.074               0.0                 0.0          1.787      1.489    0.894
        2020       0.118      0.059      0.0     0.177               0.0                 0.0          1.536      1.772    0.768
        2021       0.111      0.055      0.0     0.222               0.0                 0.0          1.552      1.774    1.442
        2022       0.161      0.

In [25]:
# This is worth internalizing as a process story too, because it's exactly the rigor these roles want: 
#"I found a dramatic result — a key risk term at zero — but before building on it I ran a robustness check with looser matching, 
# and discovered the exact-phrase version was undercounting. I revised the claim to what actually held: the accounting-classification terms, 
# which truly were absent."

In [26]:
#One small honest note on even the surviving finding, so you're fully armoured: the reason "held-to-maturity" and "available-for-sale" \
#are absent from Item 1A is partly structural — those terms live in the financial-statement notes and MD&A (Item 7/8), not usually the 
# Risk Factors narrative. So the sharpest interviewer might say "of course they're not in 1A, that's not where accounting classifications go." 
#Your reply: that's exactly the point — the Risk Factors section is supposed to translate those balance-sheet realities into forward-looking risk 
#language for investors, and SVB's didn't. But acknowledge the structural caveat rather than pretending the zero is purely damning; that honesty is 
#what makes the rest of your claims credible.

#Where we are: three features done (tone, readability, n-grams/key-terms), all pointing the same way, all now robustness-checked. Next is the 
#drift analysis — TF-IDF cosine similarity (lexical drift: how much the vocabulary changes year to year) and Sentence-BERT (semantic drift: how 
#much the meaning shifts).

# Drift analysis: TF-IDF (lexical) + Sentence-BERT (semantic)

## TF-IDF

In [27]:
# ==============================================================
# CELL 8a — Lexical drift via TF-IDF + cosine similarity
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")
corpus = corpus.sort_values(["ticker", "fiscal_year"]).reset_index(drop=True)

# --- Fit TF-IDF ONCE on the full corpus (shared vocabulary space) ---
# Design choice: retrospective descriptive study, no prediction target,
# so a shared space is correct and makes all comparisons commensurable.
# Caps keep rare one-off tokens from creating spurious drift.
vectorizer = TfidfVectorizer(
    max_features=5000,   # top 5000 terms by frequency — stable vocabulary
    min_df=3,            # a term must appear in >=3 documents to count
    max_df=0.9,          # drop terms in >90% of docs (ubiquitous boilerplate)
    ngram_range=(1, 2),  # unigrams + bigrams
)
X = vectorizer.fit_transform(corpus["text_heavy"])   # heavy text = content words
print(f"TF-IDF matrix: {X.shape[0]} documents x {X.shape[1]} features")

# --- Year-over-year lexical similarity WITHIN each bank ---
# High cosine = little drift (same vocabulary); low = big vocabulary change.
rows = []
for ticker in corpus["ticker"].unique():
    idx = corpus.index[corpus["ticker"] == ticker].tolist()
    sub = corpus.loc[idx].sort_values("fiscal_year")
    years = sub["fiscal_year"].tolist()
    mat = X[sub.index.tolist()]
    for i in range(1, len(years)):
        sim = cosine_similarity(mat[i-1], mat[i])[0, 0]
        rows.append({"ticker": ticker,
                     "year_from": years[i-1], "year_to": years[i],
                     "lexical_similarity": round(float(sim), 4),
                     "lexical_drift": round(1 - float(sim), 4)})

drift = pd.DataFrame(rows)
drift.to_csv(PROCESSED / "features_lexical_drift.csv", index=False)

# --- SVB's year-over-year lexical drift ---
print("\n" + "="*60)
print("SVB (SIVB) lexical drift year-over-year:")
print("(higher drift = bigger vocabulary change that year)")
print("="*60)
svb = drift[drift["ticker"] == "SIVB"]
print(svb[["year_from","year_to","lexical_similarity","lexical_drift"]].to_string(index=False))

# --- Is SVB's 2021->2022 drift unusual vs peers' 2021->2022? ---
print("\n" + "="*60)
print("2021->2022 lexical drift, all banks (the pre-collapse transition):")
print("="*60)
final = drift[(drift["year_from"]==2021) & (drift["year_to"]==2022)].sort_values("lexical_drift", ascending=False)
print(final[["ticker","lexical_similarity","lexical_drift"]].to_string(index=False))

svb_final = final[final["ticker"]=="SIVB"]["lexical_drift"].values[0]
peer_mean = final[final["ticker"]!="SIVB"]["lexical_drift"].mean()
print(f"\nSVB 2021->2022 drift: {svb_final:.4f}")
print(f"Peer mean 2021->2022 drift: {peer_mean:.4f}")
print(f"SVB is {'ABOVE' if svb_final > peer_mean else 'BELOW'} the peer average.")

TF-IDF matrix: 56 documents x 5000 features

SVB (SIVB) lexical drift year-over-year:
(higher drift = bigger vocabulary change that year)
 year_from  year_to  lexical_similarity  lexical_drift
      2016     2017              0.9786         0.0214
      2017     2018              0.9781         0.0219
      2018     2019              0.9048         0.0952
      2019     2020              0.7658         0.2342
      2020     2021              0.9135         0.0865
      2021     2022              0.9748         0.0252

2021->2022 lexical drift, all banks (the pre-collapse transition):
ticker  lexical_similarity  lexical_drift
    RF              0.8354         0.1646
  ZION              0.8637         0.1363
   KEY              0.9040         0.0960
   WAL              0.9272         0.0728
  PACW              0.9595         0.0405
  SIVB              0.9748         0.0252
   CMA              0.9981         0.0019
  FITB              0.9992         0.0008

SVB 2021->2022 drift: 0.0252
P

In [28]:
#Verify the COVID spike first — quick cell printing all banks' drift trajectories, to confirm 2019→2020 is panel-wide. Turns an inference into a finding.

In [29]:
# ==============================================================
# CELL 8b — Is the 2020 drift spike panel-wide? (COVID check)
# ==============================================================
import pandas as pd
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
drift = pd.read_csv(PROCESSED / "features_lexical_drift.csv")

# Pivot: rows = bank, cols = the year-transition, values = drift
drift["transition"] = drift["year_from"].astype(str) + "->" + drift["year_to"].astype(str)
grid = drift.pivot(index="ticker", columns="transition", values="lexical_drift")

pd.set_option("display.max_columns", None, "display.width", 140)
print("="*70)
print("Lexical drift grid (rows=bank, cols=year transition)")
print("Higher = bigger vocabulary change that year")
print("="*70)
print(grid.to_string())

# Column means: which transition had the most drift ACROSS all banks?
print("\n" + "-"*70)
print("Average drift across ALL banks, per transition:")
print("-"*70)
col_means = grid.mean().sort_values(ascending=False)
print(col_means.round(4).to_string())

top_transition = col_means.index[0]
print(f"\nBiggest panel-wide shift: {top_transition}")
print(f"  If this is 2019->2020 and ALL banks are elevated, that's the")
print(f"  COVID rewrite — a sector-wide event, confirming the drift metric")
print(f"  detects real language change.")

# How many banks had their PERSONAL max drift in the 2019->2020 transition?
personal_max = grid.idxmax(axis=1)
n_2020 = (personal_max == "2019->2020").sum()
print(f"\nBanks whose LARGEST drift year was 2019->2020: {n_2020} of {len(grid)}")

Lexical drift grid (rows=bank, cols=year transition)
Higher = bigger vocabulary change that year
transition  2016->2017  2017->2018  2018->2019  2019->2020  2020->2021  2021->2022
ticker                                                                            
CMA             0.0033      0.0067      0.0009      0.0032      0.0018      0.0019
FITB            0.0017      0.0033      0.0220      0.0018      0.0015      0.0008
KEY             0.0275      0.0396      0.0705      0.1044      0.0945      0.0960
PACW            0.0778      0.1045      0.0353      0.0570      0.0205      0.0405
RF              0.0257      0.0217      0.0490      0.0967      0.0720      0.1646
SIVB            0.0214      0.0219      0.0952      0.2342      0.0865      0.0252
WAL             0.0598      0.0156      0.0335      0.0954      0.3873      0.0728
ZION            0.0345      0.5353      0.0245      0.1224      0.1955      0.1363

----------------------------------------------------------------------
A

In [30]:
# This is exactly why we verified instead of asserting — my COVID hypothesis does not hold up, and I was wrong to suggest it confidently.
# Only 2 of 8 banks had their biggest drift in 2019→2020, and the panel-wide biggest shift is 2020→2021, not 2019→2020. 
# The drift is not a clean, synchronized sector-wide spike.

In [31]:
#Quick-check the ZION 2017→2018 outlier — confirm it's a real rewrite, not a parse glitch, so the peer baseline is trustworthy.
# (5 minutes, and it's the kind of due diligence that's itself a good interview point.)

In [32]:
# ==============================================================
# CELL 8c — Diagnose ZION 2017 vs 2018 (the 0.535 drift outlier)
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")

z17 = corpus[(corpus.ticker=="ZION") & (corpus.fiscal_year==2017)].iloc[0]
z18 = corpus[(corpus.ticker=="ZION") & (corpus.fiscal_year==2018)].iloc[0]

print("="*60)
print("ZION 2017 vs 2018 — basic sanity")
print("="*60)
print(f"2017: light={z17['n_words_light']:>6} words, heavy={z17['n_words_heavy']:>6} words")
print(f"2018: light={z18['n_words_light']:>6} words, heavy={z18['n_words_heavy']:>6} words")

# Compare to Zions' OTHER years — is either year an outlier in length?
print("\nZION word counts across ALL years (spot a length anomaly):")
zall = corpus[corpus.ticker=="ZION"].sort_values("fiscal_year")
print(zall[["fiscal_year","n_words_light","n_words_heavy"]].to_string(index=False))

# What words most distinguish 2017 from 2018? (TF-IDF on just these two)
print("\n" + "="*60)
print("Top distinguishing terms: what's in one year but not the other")
print("="*60)
vec = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
M = vec.fit_transform([z17["text_heavy"], z18["text_heavy"]])
terms = np.array(vec.get_feature_names_out())
diff = (M[1].toarray()[0] - M[0].toarray()[0])  # positive = more in 2018
top_2018 = terms[diff.argsort()[::-1][:15]]
top_2017 = terms[diff.argsort()[:15]]
print("More prominent in 2018:", ", ".join(top_2018))
print("\nMore prominent in 2017:", ", ".join(top_2017))

# Opening text of each — does either look truncated / wrong-section?
print("\n" + "="*60)
print("First 300 chars of each (light text) — check for truncation/garbage")
print("="*60)
print("2017:", z17["text_light"][:300])
print("\n2018:", z18["text_light"][:300])

ZION 2017 vs 2018 — basic sanity
2017: light=  5643 words, heavy=  3266 words
2018: light=  6699 words, heavy=  3852 words

ZION word counts across ALL years (spot a length anomaly):
 fiscal_year  n_words_light  n_words_heavy
        2016           5685           3283
        2017           5643           3266
        2018           6699           3852
        2019           6577           3812
        2020           7493           4279
        2021           5667           3287
        2022           5927           3435

Top distinguishing terms: what's in one year but not the other
More prominent in 2018: bank, security, national, occ, libor, act, zion bancorporation, bancorporation, regime, national bank, affect bank, sec, association subsidiary, bancorporation national, bank act

More prominent in 2017: company, capital, risk, affect company, interest, dividend, regulatory, stress, deposit, frb, subsidiary bank, assumption, risk company, subsidiary, company subsidiary

First 300 ch

In [34]:
#it's a real rewrite, not an artifact. This is a genuinely interesting finding, and the diagnostic pinned it exactly.
#the distinguishing terms confirm it: 2018 is dominated by "bank," "national bank," "bank act," "occ," "zion bancorporation," "association subsidiary,
# " "regime" — while 2017 leans on "company," "frb," "regulatory."

# What actually happened: Zions underwent a corporate/regulatory restructuring around 2018.

## Sentence BERT

In [36]:
# ==============================================================
# CELL 8d-1 — Load Sentence-BERT & test the Zions prediction
# ==============================================================
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")

# Load the model (downloads ~90 MB once, then cached locally)
print("Loading Sentence-BERT (all-MiniLM-L6-v2)... first run downloads ~90MB")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Loaded. Max sequence length:", model.max_seq_length, "tokens")

def embed_document(text_light, chunk_words=200):
    """
    Embed a long document by chunking into ~200-word pieces, embedding each,
    and averaging. This ensures the WHOLE document is represented, not just
    the first 512 tokens (the silent-truncation trap).
    """
    words = text_light.split()
    if len(words) == 0:
        return None
    chunks = [" ".join(words[i:i+chunk_words])
              for i in range(0, len(words), chunk_words)]
    chunk_embeddings = model.encode(chunks, show_progress_bar=False)
    return chunk_embeddings.mean(axis=0)  # document-level embedding

# --- Direct test: Zions 2017 vs 2018 (lexical drift was 0.53) ---
z17 = corpus[(corpus.ticker=="ZION") & (corpus.fiscal_year==2017)].iloc[0]
z18 = corpus[(corpus.ticker=="ZION") & (corpus.fiscal_year==2018)].iloc[0]

e17 = embed_document(z17["text_light"])
e18 = embed_document(z18["text_light"])
sem_sim = cosine_similarity([e17], [e18])[0,0]

print("\n" + "="*60)
print("THE ZIONS TEST — cosmetic rewrite or real change?")
print("="*60)
print(f"  Lexical similarity  (TF-IDF): 0.4647  (drift 0.5353 — HUGE)")
print(f"  Semantic similarity (SBERT) : {sem_sim:.4f}  (drift {1-sem_sim:.4f})")
print()
if sem_sim > 0.90:
    print("  => PREDICTION CONFIRMED: semantic drift is tiny despite huge")
    print("     lexical drift. The 'Company'->'Bank' rewrite was COSMETIC —")
    print("     the meaning barely changed. This is the textbook case for")
    print("     why lexical and semantic drift must both be measured.")
else:
    print(f"  => Semantic drift is also notable ({1-sem_sim:.4f}) — the change")
    print("     may be more than cosmetic. Worth a closer look.")

Loading Sentence-BERT (all-MiniLM-L6-v2)... first run downloads ~90MB


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded. Max sequence length: 256 tokens

THE ZIONS TEST — cosmetic rewrite or real change?
  Lexical similarity  (TF-IDF): 0.4647  (drift 0.5353 — HUGE)
  Semantic similarity (SBERT) : 0.9191  (drift 0.0809)

  => PREDICTION CONFIRMED: semantic drift is tiny despite huge
     lexical drift. The 'Company'->'Bank' rewrite was COSMETIC —
     the meaning barely changed. This is the textbook case for
     why lexical and semantic drift must both be measured.


In [37]:
# ==============================================================
# CELL 8d-2 — Semantic drift for all banks + lexical vs semantic
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")
corpus = corpus.sort_values(["ticker","fiscal_year"]).reset_index(drop=True)

# --- Embed ALL 56 documents (chunk + average) ---
print("Embedding all 56 documents (chunk + average)...")
embeddings = {}
for _, row in corpus.iterrows():
    key = (row["ticker"], row["fiscal_year"])
    embeddings[key] = embed_document(row["text_light"])   # reuse fn from 8d-1
print("Done embedding.\n")

# --- Year-over-year semantic similarity within each bank ---
sem_rows = []
for ticker in corpus["ticker"].unique():
    sub = corpus[corpus.ticker==ticker].sort_values("fiscal_year")
    years = sub["fiscal_year"].tolist()
    for i in range(1, len(years)):
        e_prev = embeddings[(ticker, years[i-1])]
        e_curr = embeddings[(ticker, years[i])]
        sim = cosine_similarity([e_prev], [e_curr])[0,0]
        sem_rows.append({"ticker": ticker,
                         "year_from": years[i-1], "year_to": years[i],
                         "semantic_similarity": round(float(sim),4),
                         "semantic_drift": round(1-float(sim),4)})

sem = pd.DataFrame(sem_rows)

# --- Merge with lexical drift for the side-by-side ---
lex = pd.read_csv(PROCESSED / "features_lexical_drift.csv")
merged = lex.merge(sem, on=["ticker","year_from","year_to"])
merged["lex_minus_sem"] = (merged["lexical_drift"] - merged["semantic_drift"]).round(4)
merged.to_csv(PROCESSED / "features_drift_combined.csv", index=False)

# --- SVB: lexical vs semantic drift over time ---
print("="*70)
print("SVB (SIVB): lexical vs semantic drift year-over-year")
print("="*70)
svb = merged[merged.ticker=="SIVB"]
print(svb[["year_from","year_to","lexical_drift","semantic_drift","lex_minus_sem"]].to_string(index=False))

# --- The final transition, all banks ---
print("\n" + "="*70)
print("2021->2022 (pre-collapse): lexical vs semantic drift, all banks")
print("="*70)
final = merged[(merged.year_from==2021)&(merged.year_to==2022)].sort_values("semantic_drift", ascending=False)
print(final[["ticker","lexical_drift","semantic_drift","lex_minus_sem"]].to_string(index=False))

svb_sem = final[final.ticker=="SIVB"]["semantic_drift"].values[0]
peer_sem = final[final.ticker!="SIVB"]["semantic_drift"].mean()
print(f"\nSVB 2021->2022 semantic drift: {svb_sem:.4f}")
print(f"Peer mean semantic drift:      {peer_sem:.4f}")
print(f"SVB is {'ABOVE' if svb_sem>peer_sem else 'BELOW'} peer average on MEANING change too.")

# Where is lexical >> semantic? (cosmetic rewrites like Zions)
print("\n" + "="*70)
print("Biggest 'cosmetic' rewrites (high lexical, low semantic drift):")
print("="*70)
cosmetic = merged.sort_values("lex_minus_sem", ascending=False).head(6)
print(cosmetic[["ticker","year_from","year_to","lexical_drift","semantic_drift","lex_minus_sem"]].to_string(index=False))

Embedding all 56 documents (chunk + average)...
Done embedding.

SVB (SIVB): lexical vs semantic drift year-over-year
 year_from  year_to  lexical_drift  semantic_drift  lex_minus_sem
      2016     2017         0.0214          0.0026         0.0188
      2017     2018         0.0219          0.0019         0.0200
      2018     2019         0.0952          0.0062         0.0890
      2019     2020         0.2342          0.0136         0.2206
      2020     2021         0.0865          0.0040         0.0825
      2021     2022         0.0252          0.0042         0.0210

2021->2022 (pre-collapse): lexical vs semantic drift, all banks
ticker  lexical_drift  semantic_drift  lex_minus_sem
    RF         0.1646          0.0180         0.1466
   KEY         0.0960          0.0109         0.0851
   WAL         0.0728          0.0094         0.0634
  ZION         0.1363          0.0090         0.1273
   CMA         0.0019          0.0053        -0.0034
  SIVB         0.0252          0.0042

## LDA topic modelling

In [38]:
# ==============================================================
# CELL 9a — LDA topic model: fit and tune number of topics (k)
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")
corpus = corpus.sort_values(["ticker","fiscal_year"]).reset_index(drop=True)

# --- Build the document-term count matrix (LDA uses counts, not TF-IDF) ---
# Heavy text (content words), with caps to keep topics clean.
count_vec = CountVectorizer(
    max_features=2000,   # top 2000 content terms
    min_df=3,            # term in >=3 docs
    max_df=0.85,         # drop near-ubiquitous boilerplate
    ngram_range=(1,1),   # unigrams for cleaner topics
)
dtm = count_vec.fit_transform(corpus["text_heavy"])
vocab = np.array(count_vec.get_feature_names_out())
print(f"Document-term matrix: {dtm.shape[0]} docs x {dtm.shape[1]} terms\n")

# --- Choose k by a simple, defensible coherence proxy ---
# We try several k and compute the average pairwise similarity of each topic's
# top words in the term co-occurrence space. Higher = more coherent topics.
# (A lightweight coherence measure that needs no extra library.)
from sklearn.metrics.pairwise import cosine_similarity

def topic_coherence(lda_model, dtm, vocab, topn=10):
    """Average top-word co-occurrence coherence across topics (UMass-style)."""
    # Binary doc-term for co-occurrence counts
    binary = (dtm > 0).astype(int)
    co = (binary.T @ binary).toarray()  # term-term co-occurrence
    doc_freq = np.asarray(binary.sum(axis=0)).ravel()
    coherences = []
    for topic in lda_model.components_:
        top = topic.argsort()[::-1][:topn]
        score = 0; pairs = 0
        for i in range(1, len(top)):
            for j in range(i):
                wi, wj = top[i], top[j]
                # UMass: log((co-occurrence + 1) / doc_freq(wj))
                score += np.log((co[wi, wj] + 1) / (doc_freq[wj] + 1e-9))
                pairs += 1
        coherences.append(score / max(pairs,1))
    return np.mean(coherences)

print("Tuning k (number of topics) by coherence:\n")
results = {}
for k in [4, 5, 6, 7, 8, 10]:
    lda = LatentDirichletAllocation(n_components=k, random_state=42,
                                    max_iter=20, learning_method="batch")
    lda.fit(dtm)
    coh = topic_coherence(lda, dtm, vocab)
    results[k] = coh
    print(f"  k={k:>2}:  coherence = {coh:.3f}")

best_k = max(results, key=results.get)
print(f"\nBest k by coherence: {best_k}")
print("(Higher/closer-to-zero UMass coherence = more coherent topics)")

Document-term matrix: 56 docs x 2000 terms

Tuning k (number of topics) by coherence:

  k= 4:  coherence = -0.360
  k= 5:  coherence = -0.293
  k= 6:  coherence = -0.321
  k= 7:  coherence = -0.325
  k= 8:  coherence = -0.391
  k=10:  coherence = -0.371

Best k by coherence: 5
(Higher/closer-to-zero UMass coherence = more coherent topics)


In [39]:
# ==============================================================
# CELL 9b — Final LDA (k=5): topics + SVB topic-mix over time
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"

K = 5
lda = LatentDirichletAllocation(n_components=K, random_state=42,
                                max_iter=50, learning_method="batch")
doc_topics = lda.fit_transform(dtm)   # reuse dtm, count_vec, vocab, corpus from 9a

# --- Print each topic as its top words (you'll label these) ---
print("="*70)
print(f"THE {K} TOPICS (top 12 words each) — label them by theme:")
print("="*70)
topic_labels = {}
for t in range(K):
    top = lda.components_[t].argsort()[::-1][:12]
    words = ", ".join(vocab[top])
    print(f"\nTopic {t}: {words}")
    topic_labels[t] = f"Topic {t}"   # you can rename after seeing words

# --- Attach topic proportions to each document ---
topic_cols = [f"topic_{t}" for t in range(K)]
tdf = pd.DataFrame(doc_topics, columns=topic_cols)
tdf["ticker"] = corpus["ticker"].values
tdf["fiscal_year"] = corpus["fiscal_year"].values
tdf.to_csv(PROCESSED / "features_lda_topics.csv", index=False)

# --- SVB's topic mix over time ---
print("\n" + "="*70)
print("SVB (SIVB) topic mix over time (proportion of each topic):")
print("="*70)
svb = tdf[tdf.ticker=="SIVB"].sort_values("fiscal_year")
pd.set_option("display.max_columns", None, "display.width", 140)
print(svb[["fiscal_year"] + topic_cols].round(3).to_string(index=False))

# --- Which topic shifted MOST for SVB from 2016 to 2022? ---
print("\n" + "-"*70)
first = svb[svb.fiscal_year==2016][topic_cols].values[0]
last  = svb[svb.fiscal_year==2022][topic_cols].values[0]
change = last - first
print("SVB topic shift, 2016 -> 2022 (positive = grew):")
for t in range(K):
    print(f"  Topic {t}: {first[t]:.3f} -> {last[t]:.3f}  (Δ {change[t]:+.3f})")

# --- 2022 peer snapshot: is SVB's topic mix unusual? ---
print("\n" + "="*70)
print("2022 topic mix — SVB vs peer average:")
print("="*70)
y22 = tdf[tdf.fiscal_year==2022]
svb22 = y22[y22.ticker=="SIVB"][topic_cols].values[0]
peer22 = y22[y22.ticker!="SIVB"][topic_cols].mean().values
for t in range(K):
    flag = "  <-- SVB higher" if svb22[t] > peer22[t]+0.05 else ("  <-- SVB lower" if svb22[t] < peer22[t]-0.05 else "")
    print(f"  Topic {t}:  SVB {svb22[t]:.3f}   peer avg {peer22[t]:.3f}{flag}")

THE 5 TOPICS (top 12 words each) — label them by theme:

Topic 0: lease, borrower, total, million, billion, construction, commitment, frb, venture, ratio, board, land

Topic 1: fifth, comerica, bancorp, libor, content, frb, pandemic, section, network, borrower, shareholder, social

Topic 2: content, private, svb, borrower, libor, line, commitment, pandemic, offering, international, venture, initiative

Topic 3: occ, stress, zion, libor, bancorporation, committee, framework, content, association, pandemic, concentration, exchange

Topic 4: region, borrower, annual, content, preferred, pandemic, stress, libor, discussion, cybersecurity, lease, home

SVB (SIVB) topic mix over time (proportion of each topic):
 fiscal_year  topic_0  topic_1  topic_2  topic_3  topic_4
        2016      0.0      0.0      1.0      0.0      0.0
        2017      0.0      0.0      1.0      0.0      0.0
        2018      0.0      0.0      1.0      0.0      0.0
        2019      0.0      0.0      1.0      0.0     

In [40]:
#top words m company's name types things are also coming. So adding those into the stop words

In [42]:
# ==============================================================
# CELL 9c — LDA REFIT with bank-name / identity tokens removed
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

PROCESSED = PROJECT_ROOT / "data" / "processed"
corpus = pd.read_pickle(PROCESSED / "corpus_preprocessed.pkl")
corpus = corpus.sort_values(["ticker","fiscal_year"]).reset_index(drop=True)

# --- Identity + filler tokens that hijacked the topics: remove them ---
BANK_TOKENS = {
    "svb","silicon","valley","comerica","zion","zions","bancorporation",
    "bancorp","fifth","third","keycorp","key","regions","region","huntington",
    "western","alliance","pacwest","pacific","frb","occ","association",
    # generic filler that survived and adds no thematic meaning
    "content","million","billion","total","annual","section","discussion",
    "company","bank","banks","also","may","could","would","affect","result",
    "including","within","under","upon","however","material","adversely",
}
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
custom_stop = list(ENGLISH_STOP_WORDS.union(BANK_TOKENS))

count_vec = CountVectorizer(
    max_features=1500,
    min_df=5,             # term must appear in >=5 docs (more shared)
    max_df=0.75,          # drop terms in >75% of docs (boilerplate)
    ngram_range=(1,1),
    stop_words=custom_stop,
)
dtm = count_vec.fit_transform(corpus["text_heavy"])
vocab = np.array(count_vec.get_feature_names_out())
print(f"DTM after cleaning identity tokens: {dtm.shape[0]} docs x {dtm.shape[1]} terms\n")

# --- Refit LDA at k=5 ---
K = 5
lda = LatentDirichletAllocation(n_components=K, random_state=42,
                                max_iter=50, learning_method="batch")
doc_topics = lda.fit_transform(dtm)

print("="*70)
print(f"REFIT TOPICS (k={K}, top 12 words) — should now be THEMES, not banks:")
print("="*70)
for t in range(K):
    top = lda.components_[t].argsort()[::-1][:12]
    print(f"\nTopic {t}: {', '.join(vocab[top])}")

# --- Check: is any bank name still a top word? (should be none) ---
print("\n" + "-"*70)
all_top = set()
for t in range(K):
    all_top.update(vocab[lda.components_[t].argsort()[::-1][:12]])
leaked = all_top & BANK_TOKENS
print(f"Identity tokens still leaking into top words: {leaked if leaked else 'NONE — good'}")

# --- SVB topic mix over time (should now be a soft MIX, not 1.0/0.0) ---
topic_cols = [f"topic_{t}" for t in range(K)]
tdf = pd.DataFrame(doc_topics, columns=topic_cols)
tdf["ticker"] = corpus["ticker"].values
tdf["fiscal_year"] = corpus["fiscal_year"].values

print("\n" + "="*70)
print("SVB topic mix over time (should be a spread across topics now):")
print("="*70)
svb = tdf[tdf.ticker=="SIVB"].sort_values("fiscal_year")
pd.set_option("display.max_columns", None, "display.width", 140)
print(svb[["fiscal_year"]+topic_cols].round(3).to_string(index=False))

tdf.to_csv(PROCESSED / "features_lda_topics.csv", index=False)

DTM after cleaning identity tokens: 56 docs x 1500 terms

REFIT TOPICS (k=5, top 12 words) — should now be THEMES, not banks:

Topic 0: lease, venture, construction, private, california, financing, office, ratio, board, privacy, stockholder, iii

Topic 1: pandemic, home, analysis, intangible, goodwill, deferred, commodity, cybersecurity, liquidation, financing, stockholder, second

Topic 2: fraudulent, card, pandemic, lease, keybank, methodology, segment, prudential, merger, severe, life, people

Topic 3: senior, pandemic, stockholder, successful, team, california, transition, concentration, wab, bod, series, creditworthiness

Topic 4: pandemic, shareholder, transition, social, merger, theft, retail, changing, page, residential, hedge, cybersecurity

----------------------------------------------------------------------
Identity tokens still leaking into top words: NONE — good

SVB topic mix over time (should be a spread across topics now):
 fiscal_year  topic_0  topic_1  topic_2  topi

# Bootstrap Null Model

In [43]:
# ==============================================================
# CELL 10 — Bootstrap null model: is SVB's change unusual?
# ==============================================================
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = PROJECT_ROOT / "data" / "processed"
rng = np.random.default_rng(42)

# --- Load the features we'll test ---
tone   = pd.read_csv(PROCESSED / "features_tone.csv")
read   = pd.read_csv(PROCESSED / "features_readability.csv")
drift  = pd.read_csv(PROCESSED / "features_drift_combined.csv")  # has lexical+semantic

# ----------------------------------------------------------------------
# TEST TYPE A — "Is SVB's 2021->2022 DRIFT unusual vs all banks' drifts?"
# The null population = every bank's every year-over-year drift value.
# We ask: is SVB's 2021->2022 drift a typical draw, or in the tail?
# ----------------------------------------------------------------------
def bootstrap_z(null_population, observed, n_boot=10000):
    """
    Resample the null population (with replacement), each time drawing a
    value to represent 'a typical change under the null'. Compare the
    observed value to that null distribution.
    Returns z-score and two-sided percentile-based p-value.
    """
    null_pop = np.asarray(null_population, dtype=float)
    # Bootstrap distribution of single draws from the null population
    boot = rng.choice(null_pop, size=n_boot, replace=True)
    mu, sd = boot.mean(), boot.std(ddof=1)
    z = (observed - mu) / sd if sd > 0 else 0.0
    # Two-sided p: fraction of null draws at least as extreme as observed
    p = np.mean(np.abs(boot - mu) >= np.abs(observed - mu))
    return z, p, mu, sd

print("="*72)
print("BOOTSTRAP NULL TEST — is SVB's pre-collapse (2021->2022) change")
print("statistically distinguishable from normal bank-year variation?")
print("="*72)

tests = []

# --- Lexical drift ---
lex_null = drift["lexical_drift"].values  # all bank-year drifts
svb_lex  = drift[(drift.ticker=="SIVB")&(drift.year_from==2021)&(drift.year_to==2022)]["lexical_drift"].values[0]
z,p,mu,sd = bootstrap_z(lex_null, svb_lex)
tests.append(("Lexical drift 21->22", svb_lex, mu, z, p))

# --- Semantic drift ---
sem_null = drift["semantic_drift"].values
svb_sem  = drift[(drift.ticker=="SIVB")&(drift.year_from==2021)&(drift.year_to==2022)]["semantic_drift"].values[0]
z,p,mu,sd = bootstrap_z(sem_null, svb_sem)
tests.append(("Semantic drift 21->22", svb_sem, mu, z, p))

# ----------------------------------------------------------------------
# TEST TYPE B — "Is SVB's 2022 LEVEL unusual vs peers' 2022 levels?"
# Null population = all banks' 2022 value for that metric.
# ----------------------------------------------------------------------
# --- Negative tone level 2022 ---
neg_null = tone[tone.fiscal_year==2022]["negative_pct"].values
svb_neg  = tone[(tone.ticker=="SIVB")&(tone.fiscal_year==2022)]["negative_pct"].values[0]
z,p,mu,sd = bootstrap_z(neg_null, svb_neg)
tests.append(("Negative tone 2022 (level)", svb_neg, mu, z, p))

# --- Uncertainty tone level 2022 ---
unc_null = tone[tone.fiscal_year==2022]["uncertainty_pct"].values
svb_unc  = tone[(tone.ticker=="SIVB")&(tone.fiscal_year==2022)]["uncertainty_pct"].values[0]
z,p,mu,sd = bootstrap_z(unc_null, svb_unc)
tests.append(("Uncertainty tone 2022 (level)", svb_unc, mu, z, p))

# --- Readability (Fog) level 2022 ---
fog_null = read[read.fiscal_year==2022]["fog"].values
svb_fog  = read[(read.ticker=="SIVB")&(read.fiscal_year==2022)]["fog"].values[0]
z,p,mu,sd = bootstrap_z(fog_null, svb_fog)
tests.append(("Fog readability 2022 (level)", svb_fog, mu, z, p))

# --- Results table ---
print(f"\n{'Metric':<32}{'SVB':>9}{'NullMean':>10}{'z-score':>9}{'p-value':>9}  Verdict")
print("-"*80)
for name, obs, mu, z, p in tests:
    verdict = "SIGNIFICANT" if p < 0.05 else "not sig (null holds)"
    print(f"{name:<32}{obs:>9.3f}{mu:>10.3f}{z:>9.2f}{p:>9.3f}  {verdict}")

print("\n" + "="*72)
print("INTERPRETATION")
print("="*72)
print("z near 0 and p > 0.05  => SVB is a TYPICAL draw: its risk language is")
print("  statistically indistinguishable from normal bank variation. The null")
print("  of 'no early-warning signal' HOLDS for that metric.")
print("|z| large and p < 0.05 => SVB is an outlier on that metric (signal).")

BOOTSTRAP NULL TEST — is SVB's pre-collapse (2021->2022) change
statistically distinguishable from normal bank-year variation?

Metric                                SVB  NullMean  z-score  p-value  Verdict
--------------------------------------------------------------------------------
Lexical drift 21->22                0.025     0.069    -0.47    0.543  not sig (null holds)
Semantic drift 21->22               0.004     0.011    -0.42    0.416  not sig (null holds)
Negative tone 2022 (level)          4.322     4.167     0.23    1.000  not sig (null holds)
Uncertainty tone 2022 (level)       3.613     3.665    -0.08    0.877  not sig (null holds)
Fog readability 2022 (level)       24.012    23.234     1.12    0.497  not sig (null holds)

INTERPRETATION
z near 0 and p > 0.05  => SVB is a TYPICAL draw: its risk language is
  statistically indistinguishable from normal bank variation. The null
  of 'no early-warning signal' HOLDS for that metric.
|z| large and p < 0.05 => SVB is an outli